### Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sn

print(pg.__version__) # 0.5.3
print(pd.__version__) # 2.0.3
print(np.__version__) # 1.24.3
print(sn.__version__) # 0.13.0

from utils_MS import *

# %load_ext autotime

In [ ]:
# def run(exp):

### Parameters

In [ ]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"]

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

apply_transformation = params["apply_transformation"]
print("Apply transformation:", apply_transformation)

threshold_corr = params["threshold_corr"]
print("Threshold corr:\t", threshold_corr)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups_id_no = params["groups_id_no"]
print("Groups id (no):\t", groups_id_no)

In [ ]:
# Remove
# groups_id = ["LSNB", "BC", "RCC", "BPH", "PD"] # "OSA" Memory issue
# groups_id = ["AR", "BC"]

### Load dataset

In [ ]:
# read raw data
df_join_raw = pd.read_csv("experiments/input/{}_raw.csv".format(exp), index_col=0)
df_join_raw

In [ ]:
# get metadata
df_join_raw_metadata = df_join_raw.iloc[:, :2]
df_join_raw_metadata

In [ ]:
# filter by samples
columns_sample = [column for column in df_join_raw.columns if column.split("_")[0] not in groups_id_no]
df_join_raw_intensity = df_join_raw.loc[:, columns_sample]
df_join_raw_intensity = df_join_raw_intensity.iloc[:, 3:]
df_join_raw_intensity

In [ ]:
df_join_raw_intensity.info()

In [ ]:
check_dataset(df_join_raw_intensity)

### Generate graphs

In [ ]:
# Transformation (log10)

if apply_transformation:
	df_join_raw_log = log10_global(df_join_raw_intensity)
else:
	df_join_raw_log = df_join_raw_intensity.copy()
df_join_raw_log.head()

In [ ]:
check_dataset(df_join_raw_log)

In [ ]:
# split graph in groups and subgroups

""" def split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id, by_group=False):
	list_df_groups_subgroups = []
	for group in groups_id:
		df_aux = df_join_raw_log.filter(like=group)
		list_aux = []
		
		if by_group:
			list_aux.append(df_aux)
		else:
			for subgroup in subgroups_id[group]:
				list_aux.append(df_aux.filter(like="{}_{}.".format(group, subgroup)))
		list_df_groups_subgroups.append(list_aux)
	return list_df_groups_subgroups """

dict_df_groups_subgroups = split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id)
dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

In [ ]:
check_dataset(dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]])

In [ ]:
# No apply transpose for kneighbors_graph
# dict_groups_subgroups_t = dict_df_groups_subgroups.copy()

# Aplly Transpose
dict_groups_subgroups_t = transpose_global(dict_df_groups_subgroups)

dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

In [ ]:
check_dataset(dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]])

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import kneighbors_graph

def correlation_ledoitwolf_global(exp, dict_groups_subgroups_t):
	dict_groups_subgroups_t_corr = {}
	for group_id, dict_groups in dict_groups_subgroups_t.items():
		dict_aux = {}
		for subgroup_id, df_subgroup in dict_groups.items():
			print(group_id, subgroup_id, df_subgroup.shape)
			
			""" import numpy as np
			cov = np.cov(df_subgroup.values, rowvar=False)
			cond = np.linalg.cond(cov)
			print("Condition number:", cond, cond > 1e8) # ill-conditioned if > 1e8 (True, instable) """

			scaler = StandardScaler()
			df_subgroup_scaled = scaler.fit_transform(df_subgroup)
			lw = LedoitWolf()
			lw.fit(df_subgroup_scaled)

			# Matriz de covarianza regularizada
			cov = lw.covariance_
			std = np.sqrt(np.diag(cov))
			corr = cov / np.outer(std, std)
			matrix = pd.DataFrame(corr)
		
			dict_aux[subgroup_id] = matrix
			
			matrix.to_csv("experiments/output/{}/correlations/{}_{}.csv".format(exp, group_id, subgroup_id), index=True)
		dict_groups_subgroups_t_corr[group_id] = dict_aux
	return dict_groups_subgroups_t_corr

def correlation_kneighbors_graph_global(exp, dict_groups_subgroups_t):
	dict_groups_subgroups_t_corr = {}
	for group_id, dict_groups in dict_groups_subgroups_t.items():
		dict_aux = {}
		for subgroup_id, df_subgroup in dict_groups.items():
			print(group_id, subgroup_id, df_subgroup.shape)
			
			""" import numpy as np
			cov = np.cov(df_subgroup.values, rowvar=False)
			cond = np.linalg.cond(cov)
			print("Condition number:", cond, cond > 1e8) # ill-conditioned if > 1e8 (True, instable) """
			
			k = 10
			scaler = StandardScaler()
			df_subgroup_scaled = scaler.fit_transform(df_subgroup)
			A = kneighbors_graph(
				df_subgroup_scaled, # X, X_scaled
				n_neighbors=k,
				metric="euclidean", # "cosine",
				mode="distance",
				include_self=True
			)
			matrix = pd.DataFrame(A.toarray())
		
			dict_aux[subgroup_id] = matrix
			
			matrix.to_csv("experiments/output/{}/correlations/{}_{}.csv".format(exp, group_id, subgroup_id), index=True)
		dict_groups_subgroups_t_corr[group_id] = dict_aux
	return dict_groups_subgroups_t_corr

In [17]:
# Correlation matrix (partial correlation)

# Option 1
# dict_groups_subgroups_t_corr = correlation_global(exp, dict_groups_subgroups_t)

# Option 2
dict_groups_subgroups_t_corr = correlation_ledoitwolf_global(exp, dict_groups_subgroups_t)

# Option 3
# dict_groups_subgroups_t_corr = correlation_kneighbors_graph_global(exp, dict_groups_subgroups_t)

dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

disease 10 (241, 5444)
disease 11 (23, 5444)
disease 12 (69, 5444)
disease 13 (116, 5444)
disease 14 (106, 5444)
disease 15 (103, 5444)
disease 16 (143, 5444)
disease 17 (87, 5444)
disease 18 (105, 5444)
disease 2 (83, 5444)
disease 3 (78, 5444)
disease 4 (232, 5444)
disease 5 (121, 5444)
disease 6 (178, 5444)
disease 7 (85, 5444)
disease 8 (75, 5444)
disease 9 (28, 5444)


,0,1,2,3,4,5,6,7,8,9,...,5434,5435,5436,5437,5438,5439,5440,5441,5442,5443
0,1.000000,0.174581,0.165038,0.212890,0.159773,0.176521,0.182648,0.181443,0.169069,0.188352,...,0.143699,0.122297,0.111452,0.039771,0.138912,0.100034,0.199383,0.156175,0.147559,0.160229
1,0.174581,1.000000,0.086543,0.247080,0.390980,0.393033,0.383695,0.350129,0.335462,0.278491,...,0.265565,0.274280,0.160716,0.236543,0.236555,0.246695,0.356938,0.250359,0.303985,0.267378
2,0.165038,0.086543,1.000000,0.097880,0.069158,0.065091,0.046552,0.044415,0.018479,0.024209,...,0.155852,0.125760,0.194264,0.143943,0.112840,0.077547,0.095005,0.132008,0.150810,0.098847
3,0.212890,0.247080,0.097880,1.000000,0.462738,0.473673,0.453659,0.485622,0.458887,0.453186,...,0.334667,0.339696,0.158398,0.201391,0.276562,0.249136,0.331116,0.338701,0.349476,0.275583
4,0.159773,0.390980,0.069158,0.462738,1.000000,0.787010,0.753084,0.695534,0.633428,0.592446,...,0.245584,0.229727,0.037284,0.127705,0.223469,0.189945,0.305924,0.244023,0.240023,0.216285


In [18]:
# Check correlation matrices

dict_groups_subgroups_t_corr

{'disease': {'1':           0         1         2         3         4         5         6     \
  0     1.000000  0.174581  0.165038  0.212890  0.159773  0.176521  0.182648   
  1     0.174581  1.000000  0.086543  0.247080  0.390980  0.393033  0.383695   
  2     0.165038  0.086543  1.000000  0.097880  0.069158  0.065091  0.046552   
  3     0.212890  0.247080  0.097880  1.000000  0.462738  0.473673  0.453659   
  4     0.159773  0.390980  0.069158  0.462738  1.000000  0.787010  0.753084   
  ...        ...       ...       ...       ...       ...       ...       ...   
  5439  0.100034  0.246695  0.077547  0.249136  0.189945  0.186858  0.187776   
  5440  0.199383  0.356938  0.095005  0.331116  0.305924  0.301003  0.282564   
  5441  0.156175  0.250359  0.132008  0.338701  0.244023  0.248213  0.241919   
  5442  0.147559  0.303985  0.150810  0.349476  0.240023  0.236497  0.240413   
  5443  0.160229  0.267378  0.098847  0.275583  0.216285  0.182179  0.172228   
  
            7        

In [19]:
check_dataset(dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][1]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 7535862
Count zero:	 5612
Count positive:	 22095662
Count greater than 1:	 0
Count less than -1:	 0


In [20]:
def build_graph_weight_global_directed_new(exp, dict_groups_subgroups_t_corr, threshold=0.3):
	dict_groups_subgroups_t_corr_g = {}
	for group_id, dict_groups in dict_groups_subgroups_t_corr.items():
		dict_aux = {}
		for subgroup_id, df_subgroup in dict_groups.items():
			# Percentil Correlaciones conservadas 	Densidad
			# 50	    50 %	                    Muy densa
            # 75	    25 %	                    Densa
            # 90	    10 %	                    Moderadamente dispersa
            # 95	    5 %	                        Dispersa
            # 99	    1 %	                        Muy dispersa
			threshold = np.percentile(np.abs(df_subgroup), 90)
			
			df_weighted_edges = (df_subgroup.where(np.triu(np.ones(df_subgroup.shape), k=1).astype(bool)).stack())
			df_weighted_edges = df_weighted_edges.dropna().to_frame()
			df_weighted_edges.reset_index(inplace=True)
			df_weighted_edges.columns = ["source", "target", "weight"]
			df_weighted_edges = df_weighted_edges[df_weighted_edges["weight"].abs() >= threshold]
			df_weighted_edges["subgroup"] = [subgroup_id] * len(df_weighted_edges)
			dict_aux[subgroup_id] = df_weighted_edges
			
			df_weighted_edges.to_csv("experiments/output/{}/preprocessing/edges/{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# G = nx.from_pandas_edgelist(df_weighted_edges, "source", "target", edge_attr=["weight"])
			# print(groups_id[i], subgroups_id[groups_id[i]][j], G.number_of_nodes(), G.number_of_edges())
			# nx.write_gexf(G, "experiments/output/{}/preprocessing/graphs/graphs_{}_{}.gexf".format(exp, groups_id[i], subgroups_id[groups_id[i]][j]))
		dict_groups_subgroups_t_corr_g[group_id] = dict_aux
	return dict_groups_subgroups_t_corr_g

In [21]:
# Build graph (corpus graphs)

# dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed_new(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,source,target,weight,subgroup
42,0,43,0.513130,1
49,0,50,0.540938,1
50,0,51,0.580704,1
58,0,59,0.531386,1
63,0,64,0.632894,1


In [22]:
def create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups, df_join_raw_metadata):	
	for group_id in tqdm(groups_id):
		for subgroup_id in tqdm(subgroups_id[group_id]):
			df_weighted_edges = pd.read_csv("experiments/output/{}/preprocessing/edges/{}_{}.csv".format(exp, group_id, subgroup_id))
			# print(df_weighted_edges)
			G = nx.from_pandas_edgelist(df_weighted_edges, "source", "target", edge_attr=["weight", "subgroup"])
			dict_id_idx = dict(zip(list(G.nodes()), range(G.number_of_nodes())))
			G = nx.relabel_nodes(G, dict_id_idx)

			df_nodes = dict_df_groups_subgroups[group_id][subgroup_id].loc[list(dict_id_idx.keys())] # A_1.1, A_1.2, A_1.3
			# from IPython.display import display
			# display(df_nodes)

			# nodes, with node features
			metadata = df_join_raw_metadata.loc[df_nodes.index] # Average Rt, Average Mz
			# intensity = df_join_raw_log.loc[df_nodes.index] # A_1.1, A_1.2, A_1.3, A_2.1, ...

			""" e = 1e-8
			mz = metadata.iloc[:, 1].values
			rt = metadata.iloc[:, 0].values
			intensity_mean = df_nodes.mean(axis=1).values
			intensity_std = df_nodes.std(axis=1).values
			intensity_cv = intensity_std / intensity_mean
			presence_ratio = (df_nodes > 0).mean(axis=1)

			mz_log = np.log10(mz + e)
			# intensity_mean_log = np.log10(intensity_mean + e)
			
			# mz_z = (mz_log - mz_log.mean()) / mz_log.std() # z-score
			rt_z = (rt - rt.mean()) / rt.std() # z-score
			# intensity_mean_z = (intensity_mean_log - intensity_mean_log.mean()) / intensity_mean_log.std() # z-score

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": mz_log,
				"rt": rt_z,
				"intensity_mean": intensity_mean,
				"intensity_std": intensity_std,
				"intensity_cv": intensity_cv,
				"presence_ratio": presence_ratio
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			# df_node_features.insert(0, "idx", list(dict_id_idx.values()))
			# df_node_features.insert(1, "id", list(dict_id_idx.keys()))
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features) """

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": metadata.iloc[:, 1].values,
				"rt": metadata.iloc[:, 0].values,
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features)

			# edges
			edges = list(G.edges())
			df_edges = pd.DataFrame(edges, columns=["source", "target"])
			df_edges["weight"] = [G.get_edge_data(*edge)["weight"] for edge in edges]
			df_edges["subgroup"] = [G.get_edge_data(*edge)["subgroup"] for edge in edges]
			df_edges.to_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)


In [23]:
# create dataset - nodes/edge data for PyTorch Geometric/DGL framework

# IMPORTANT
dict_df_groups_subgroups_ = split_groups_subgroups(df_join_raw_intensity, groups_id, subgroups_id) # Important (intesities without Log)

for data_variation in data_variations:
	if data_variation == "none":
		create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, df_join_raw_metadata)
	else:
		# dynamic graph to static graph
		create_graph_data_directed_variation(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, data_variation)

100%|██████████| 1/1 [02:25<00:00, 145.07s/it]


In [24]:
# details
list_details = []
	
for group_id in groups_id:
	subgroups_id_ = []
	for data_variation in data_variations:
		if data_variation == "none":
			subgroups_id_ += subgroups_id[group_id]
		else:
			subgroups_id_ += [data_variation]
	# print(subgroups)
	
	for subgroup_id_ in subgroups_id_:
		try:
			df_edges = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id_))

			G = nx.from_pandas_edgelist(df_edges.iloc[:, [0, 1]])
			list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), nx.density(G), np.nan, nx.is_connected(G)])
		except:
			list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), np.nan, np.nan, np.nan])

df_details = pd.DataFrame(list_details, columns=["Group", "Subgroup", "Num. nodes", "Num. edges", "Density", "Diameter", "Is connected"])
df_details.to_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp), index=False)

df_details = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp))
df_details

,Group,Subgroup,Num. nodes,Num. edges,Density,Diameter,Is connected
0,disease,1,5346,1479135,0.103529,NaN,False
1,disease,10,5392,1479135,0.101770,NaN,False
2,disease,11,5444,1479135,0.099835,NaN,True
3,disease,12,5417,1479135,0.100832,NaN,False
4,disease,13,5438,1479135,0.100055,NaN,False
5,disease,14,5444,1479135,0.099835,NaN,True
6,disease,15,5431,1479135,0.100313,NaN,True
7,disease,16,5438,1479135,0.100055,NaN,True
8,disease,17,5412,1479135,0.101019,NaN,True
9,disease,18,5438,1479135,0.100055,NaN,True
